In [7]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

RANDOM_STATE = 42
TARGET = "bilissel_performans_skoru"
N_SPLITS = 3


In [8]:
def find_file(filename):
    candidate_paths = [
        filename,
        f"./{filename}",
        f"../input/{filename}",
        f"/kaggle/input/{filename}",
        f"../{filename}",
    ]

    for path in candidate_paths:
        if os.path.exists(path):
            return path

    kaggle_root = "/kaggle/input"
    if os.path.exists(kaggle_root):
        for root, _, files in os.walk(kaggle_root):
            if filename in files:
                return os.path.join(root, filename)

    raise FileNotFoundError(f"Dosya bulunamadı: {filename}")

train_path = find_file("train.csv")
test_path = find_file("test_x.csv")
sample_path = find_file("sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("train:", train.shape)
print("test:", test.shape)
print("sample_submission:", sample_submission.shape)


train: (56000, 24)
test: (24000, 23)
sample_submission: (2, 2)


In [9]:
print("Train kolonları:")
print(train.columns.tolist())

print("Target özeti:")
print(train[TARGET].describe())

missing_summary = pd.DataFrame({
    "train_missing_count": train.isna().sum(),
    "train_missing_ratio": train.isna().mean(),
    "test_missing_count": test.isna().sum(),
    "test_missing_ratio": test.isna().mean(),
}).sort_values("train_missing_ratio", ascending=False)

missing_summary[missing_summary.sum(axis=1) > 0]


Train kolonları:
['id', 'yas', 'cinsiyet', 'meslek', 'vucut_kitle_indeksi', 'ulke', 'rem_yuzdesi', 'derin_uyku_yuzdesi', 'uykuya_dalma_suresi_dk', 'gecelik_uyanma_sayisi', 'uyku_oncesi_kafein_mg', 'uyku_oncesi_ekran_suresi_dk', 'gunluk_adim_sayisi', 'sekerleme_suresi_dk', 'stres_skoru', 'gunluk_calisma_saati', 'kronotip', 'ruh_sagligi_durumu', 'dinlenik_nabiz_bpm', 'oda_sicakligi_celsius', 'hafta_sonu_uyku_farki_saat', 'mevsim', 'gun_tipi', 'bilissel_performans_skoru']
Target özeti:
count    56000.000000
mean         5.913096
std          2.231759
min          0.000000
25%          4.397431
50%          6.032249
75%          7.574980
max         10.000000
Name: bilissel_performans_skoru, dtype: float64


,train_missing_count,train_missing_ratio,test_missing_count,test_missing_ratio
kronotip,1968,0.035143,832.0,0.034667
vucut_kitle_indeksi,1752,0.031286,648.0,0.027000
stres_skoru,1715,0.030625,765.0,0.031875
uyku_oncesi_kafein_mg,1463,0.026125,697.0,0.029042
meslek,1378,0.024607,622.0,0.025917
ruh_sagligi_durumu,1096,0.019571,504.0,0.021000


In [10]:
base_features = [c for c in train.columns if c not in ["id", TARGET]]
numeric_cols = train[base_features].select_dtypes(exclude="object").columns.tolist()
categorical_cols = train[base_features].select_dtypes(include="object").columns.tolist()

print("Sayısal kolon sayısı:", len(numeric_cols))
print(numeric_cols)
print("Kategorik kolon sayısı:", len(categorical_cols))
print(categorical_cols)


Sayısal kolon sayısı: 15
['yas', 'vucut_kitle_indeksi', 'rem_yuzdesi', 'derin_uyku_yuzdesi', 'uykuya_dalma_suresi_dk', 'gecelik_uyanma_sayisi', 'uyku_oncesi_kafein_mg', 'uyku_oncesi_ekran_suresi_dk', 'gunluk_adim_sayisi', 'sekerleme_suresi_dk', 'stres_skoru', 'gunluk_calisma_saati', 'dinlenik_nabiz_bpm', 'oda_sicakligi_celsius', 'hafta_sonu_uyku_farki_saat']
Kategorik kolon sayısı: 7
['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']


In [11]:
def add_features(df):
    df = df.copy()
    eps = 1e-6

    df["ulke_norm"] = df["ulke"].replace({
        "South Korea": "Guney Kore",
        "Spain": "Ispanya",
        "Sweden": "Isvec",
        "Mexico": "Meksika",
        "Netherlands": "Hollanda",
    })
    df["meslek_norm"] = df["meslek"].replace({"Lawyer": "Avukat"})

    df["rem_derin_toplam"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]
    df["hafif_uyku_yuzdesi"] = 100 - df["rem_yuzdesi"] - df["derin_uyku_yuzdesi"]
    df["rem_derin_oran"] = df["rem_yuzdesi"] / (df["derin_uyku_yuzdesi"] + eps)
    df["derin_rem_oran"] = df["derin_uyku_yuzdesi"] / (df["rem_yuzdesi"] + eps)
    df["uyku_kalitesi_raw"] = 0.45 * df["rem_yuzdesi"] + 0.55 * df["derin_uyku_yuzdesi"]
    df["uyku_bolunme_skoru"] = df["uykuya_dalma_suresi_dk"] + 10 * df["gecelik_uyanma_sayisi"]
    df["iyi_uyku_indeksi"] = (
        df["rem_yuzdesi"]
        + df["derin_uyku_yuzdesi"]
        - 0.25 * df["uykuya_dalma_suresi_dk"]
        - 2 * df["gecelik_uyanma_sayisi"]
    )
    df["uyku_verimlilik_proxy"] = (
        df["rem_derin_toplam"]
        - 0.10 * df["hafif_uyku_yuzdesi"]
        - 2 * df["gecelik_uyanma_sayisi"]
    )

    df["kafein_ekran_toplam"] = df["uyku_oncesi_kafein_mg"] / 50 + df["uyku_oncesi_ekran_suresi_dk"] / 60
    df["kafein_ekran_carpim"] = df["uyku_oncesi_kafein_mg"] * df["uyku_oncesi_ekran_suresi_dk"]
    df["aktivite_calisma_oran"] = df["gunluk_adim_sayisi"] / (df["gunluk_calisma_saati"] + 1)
    df["adim_bin_proxy"] = df["gunluk_adim_sayisi"] / 1000
    df["sekerleme_saat"] = df["sekerleme_suresi_dk"] / 60

    df["stres_calisma"] = df["stres_skoru"] * df["gunluk_calisma_saati"]
    df["stres_uyku_bolunme"] = df["stres_skoru"] * df["gecelik_uyanma_sayisi"]
    df["stres_kafein"] = df["stres_skoru"] * df["uyku_oncesi_kafein_mg"]
    df["stres_ekran"] = df["stres_skoru"] * df["uyku_oncesi_ekran_suresi_dk"]
    df["nabiz_stres"] = df["dinlenik_nabiz_bpm"] * df["stres_skoru"]
    df["saglik_risk_indeksi"] = df["stres_skoru"] + df["vucut_kitle_indeksi"] / 10 + df["dinlenik_nabiz_bpm"] / 20
    df["bmi_nabiz"] = df["vucut_kitle_indeksi"] * df["dinlenik_nabiz_bpm"]
    df["yas_stres"] = df["yas"] * df["stres_skoru"]
    df["yas_nabiz"] = df["yas"] * df["dinlenik_nabiz_bpm"]

    df["hafta_sonu_mu"] = (df["gun_tipi"] == "Hafta sonu").astype(int)
    df["sicaklik_idealden_sapma"] = (df["oda_sicakligi_celsius"] - 20).abs()
    df["sicaklik_idealden_sapma2"] = df["sicaklik_idealden_sapma"] ** 2
    df["hafta_sonu_farki_abs"] = df["hafta_sonu_uyku_farki_saat"].abs()
    df["hafta_sonu_farki2"] = df["hafta_sonu_uyku_farki_saat"] ** 2

    for col in [
        "stres_skoru",
        "uyku_oncesi_ekran_suresi_dk",
        "uyku_oncesi_kafein_mg",
        "gunluk_calisma_saati",
        "gecelik_uyanma_sayisi",
        "uykuya_dalma_suresi_dk",
    ]:
        df[f"{col}_sq"] = df[col] ** 2
        df[f"{col}_log1p"] = np.log1p(df[col])

    df["kronotip_gun_tipi"] = df["kronotip"].astype(str) + "_" + df["gun_tipi"].astype(str)
    df["ruh_kronotip"] = df["ruh_sagligi_durumu"].astype(str) + "_" + df["kronotip"].astype(str)
    df["meslek_gun_tipi"] = df["meslek_norm"].astype(str) + "_" + df["gun_tipi"].astype(str)
    df["ulke_mevsim"] = df["ulke_norm"].astype(str) + "_" + df["mevsim"].astype(str)
    df["cinsiyet_kronotip"] = df["cinsiyet"].astype(str) + "_" + df["kronotip"].astype(str)

    for col in ["meslek", "vucut_kitle_indeksi", "uyku_oncesi_kafein_mg", "stres_skoru", "kronotip", "ruh_sagligi_durumu"]:
        df[f"{col}_missing"] = df[col].isna().astype(int)

    return df

train_fe = add_features(train)
test_fe = add_features(test)

print("Feature engineering sonrası train:", train_fe.shape)
print("Feature engineering sonrası test:", test_fe.shape)


Feature engineering sonrası train: (56000, 76)
Feature engineering sonrası test: (24000, 75)


In [12]:
features = [col for col in train_fe.columns if col not in ["id", TARGET]]

X = train_fe[features].copy()
y = train_fe[TARGET].copy()
X_test = test_fe[features].copy()

cat_cols = X.select_dtypes(include="object").columns.tolist()

for col in cat_cols:
    X[col] = X[col].astype(str).fillna("__MISSING__")
    X_test[col] = X_test[col].astype(str).fillna("__MISSING__")

cat_features = [X.columns.get_loc(col) for col in cat_cols]

print("Feature sayısı:", len(features))
print("Kategorik feature sayısı:", len(cat_cols))


Feature sayısı: 74
Kategorik feature sayısı: 14


In [13]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []
models = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostRegressor(
        iterations=1600,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=RANDOM_STATE + fold,
        od_type="Iter",
        od_wait=100,
        verbose=200,
        allow_writing_files=False,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
    )

    valid_pred = np.clip(model.predict(X_valid), 0, 10)
    test_pred = np.clip(model.predict(X_test), 0, 10)

    oof_preds[valid_idx] = valid_pred
    test_preds += test_pred / N_SPLITS
    models.append(model)

    fold_rmse = rmse(y_valid, valid_pred)
    fold_scores.append(fold_rmse)
    print(f"Fold {fold} RMSE: {fold_rmse:.5f}")

cv_rmse = rmse(y, oof_preds)
print("Fold skorları:", [round(score, 5) for score in fold_scores])
print(f"Genel CV RMSE: {cv_rmse:.5f}")


0:	learn: 2.1960272	test: 2.1930883	best: 2.1930883 (0)	total: 78.3ms	remaining: 2m 5s
200:	learn: 1.2177878	test: 1.2357881	best: 1.2357881 (200)	total: 4.57s	remaining: 31.8s
400:	learn: 1.1978476	test: 1.2251892	best: 1.2251892 (400)	total: 9.29s	remaining: 27.8s
600:	learn: 1.1876015	test: 1.2231573	best: 1.2231324 (594)	total: 13.7s	remaining: 22.8s
800:	learn: 1.1766952	test: 1.2222719	best: 1.2222417 (793)	total: 18.5s	remaining: 18.4s
1000:	learn: 1.1654212	test: 1.2219522	best: 1.2218964 (977)	total: 23s	remaining: 13.8s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.221891874
bestIteration = 1010

Shrink model to first 1011 iterations.
Fold 1 RMSE: 1.22183
0:	learn: 2.1953325	test: 2.1935114	best: 2.1935114 (0)	total: 27.7ms	remaining: 44.3s
200:	learn: 1.2274632	test: 1.2162551	best: 1.2162551 (200)	total: 5s	remaining: 34.8s
400:	learn: 1.2075719	test: 1.2063462	best: 1.2063462 (400)	total: 9.89s	remaining: 29.6s
600:	learn: 1.1947629	test: 1.2041928	b

In [14]:
feature_importance = np.mean(
    [model.get_feature_importance() for model in models],
    axis=0,
)

importance_df = pd.DataFrame({
    "feature": features,
    "importance": feature_importance,
}).sort_values("importance", ascending=False)

importance_df.head(25)


,feature,importance
16,ruh_sagligi_durumu,18.296408
30,iyi_uyku_indeksi,16.856681
65,meslek_gun_tipi,7.613961
5,rem_yuzdesi,6.360561
51,stres_skoru_sq,5.895547
2,meslek,5.772146
13,stres_skoru,5.114038
52,stres_skoru_log1p,4.885670
21,gun_tipi,4.819131
31,uyku_verimlilik_proxy,4.426318


In [21]:
# sadece idleri çek
submission = test[["id"]].copy()
submission[TARGET] = np.clip(test_preds, 0, 10)

assert list(submission.columns) == ["id", TARGET]
assert len(submission) == len(test)

submission_path = "catboost_submission.csv"
submission.to_csv(submission_path, index=False)

importance_df.to_csv("feature_importance.csv", index=False)

print("Submission dosyası:", submission_path)
print("Satır sayısı:", len(submission))
submission.head()


Submission dosyası: catboost_submission.csv
Satır sayısı: 24000


,id,bilissel_performans_skoru
0,1,6.103681
1,2,6.648933
2,3,3.086844
3,4,7.227718
4,5,3.588791
